# Quantitative Contrarian Trading & Behavioural Research — End-to-End Pipeline

This notebook is the clean public-facing inference pipeline for the project.

It takes raw historical and new trade files, reconstructs the chronological pre-trade behavioural features used by the selected model, loads the frozen Experiment 10B LightGBM regressor, applies the frozen decision threshold, and exports one fade decision per new trade.

**Pipeline**

`raw trades → chronological preprocessing → behavioural feature engineering → leakage checks → frozen LightGBM → fade decision → CSV export`

The notebook does **not** retrain the model or recalibrate the threshold. The detailed research and model-development process is preserved in `01_stage1_eda.ipynb`, `02_stage2_feature_research.ipynb`, and `03_stage3_model_building.ipynb`.

## 1. Configuration

Expected repository layout:

```text
project/
├── notebooks/
│   ├── 01_stage1_eda.ipynb
│   ├── 02_stage2_feature_research.ipynb
│   ├── 03_stage3_model_building.ipynb
│   └── 04_end_to_end_pipeline.ipynb
├── data/
│   ├── User Trades/
│   └── Unseen User Trades/
├── models/
│   ├── exp10b_model.joblib
│   └── exp10b_model_metadata.json
├── results/
│   ├── backtest/
│   ├── trials/
│   ├── validation/
│   └── inference/
├── requirements.txt
└── README.md
```

The `results/inference/` directory is created automatically when predictions are exported.

In [1]:
from pathlib import Path
import json
import re

import joblib
import numpy as np
import pandas as pd


# ============================================================
# PROJECT PATHS
# ============================================================

def find_project_root():
    """Locate the repository root from either the root or notebooks folder."""
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    # Useful fallback when the notebook is opened directly from notebooks/.
    if current.name == "notebooks":
        return current.parent

    return current


PROJECT_ROOT = find_project_root()

# Historical trades available before the evaluation period.
HISTORICAL_TRADES_DIR = (
    PROJECT_ROOT
    / "data"
    / "User Trades"
)

# New / unseen trades to score.
UNSEEN_TRADES_DIR = (
    PROJECT_ROOT
    / "data"
    / "Unseen User Trades"
)

# Frozen Experiment 10B model and metadata.
FROZEN_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "exp10b_model.joblib"
)

FROZEN_MODEL_METADATA_PATH = (
    PROJECT_ROOT
    / "models"
    / "exp10b_model_metadata.json"
)

# Inference results.
PREDICTION_OUTPUT_PATH = (
    PROJECT_ROOT
    / "results"
    / "inference"
    / "unseen_trade_decisions.csv"
)


# ============================================================
# FIXED CONSTANTS
# ============================================================

RANDOM_SEED = 42

STARTING_BALANCE = 5_000.0
MAX_DRAWDOWN_RATE = 0.04

DRAWDOWN_LIMIT_AMOUNT = (
    STARTING_BALANCE
    * MAX_DRAWDOWN_RATE
)

REALIZED_DRAWDOWN_BOUNDARY = (
    -DRAWDOWN_LIMIT_AMOUNT
)

np.random.seed(
    RANDOM_SEED
)

print(
    "Project root:",
    PROJECT_ROOT,
)


Project root: /Users/charltonsiaw/Desktop/C22-veNTUre


### Verify required inputs

In [2]:
required_paths = {
    "historical trade directory":
        HISTORICAL_TRADES_DIR,
    "unseen trade directory":
        UNSEEN_TRADES_DIR,
    "frozen model":
        FROZEN_MODEL_PATH,
    "model metadata":
        FROZEN_MODEL_METADATA_PATH,
}

missing_paths = [
    f"{name}: {path}"
    for name, path in required_paths.items()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required paths are missing:\n- "
        + "\n- ".join(missing_paths)
        + "\n\nSee README.md for the expected repository structure."
    )

print(
    "Input and model paths verified."
)


Input and model paths verified.


## 2. Stage 1 — Chronological Trade Preprocessing

The following functions standardise raw trade files, preserve campaign chronology, attach information from previously completed trades, and assign trade ideas.

Only information available before each trade opens is used when constructing model inputs.

In [3]:
FILENAME_PATTERN = re.compile(
    r"Campaign[_\s]*(\d+)"
    r"[_\s]*Data[_\s]*"
    r"(\d{1,2}[_\s]*[A-Za-z]+[_\s]*\d{4})"
    r".*?"
    r"(Traders[\s_]*only|XAUUSD[\s_]*only)",
    re.IGNORECASE,
)


TRADE_ID_COLUMNS = [
    "account_id",
    "close_trade_id",
    "position_id",
    "close_order_id",
    "open_order_id",
    "user_group_id",
]


TRADE_DATETIME_COLUMNS = [
    "open_date_time",
    "close_date_time",
]


TRADE_NUMERIC_COLUMNS = [
    "lot_size",
    "duration_sec",
    "profit",
    "reverse_profit",
    "net_profit",
    "commission",
    "swap",
    "amount",
    "open_price",
    "close_price",
    "sl_price",
    "tp_price",
    "open_trade_cross_price",
    "close_trade_cross_price",
]


TRADE_COLUMN_ALIASES = {
    "account_id": "account_id",
    "accountid": "account_id",

    "close_trade_id": "close_trade_id",
    "closetradeid": "close_trade_id",

    "position_id": "position_id",
    "positionid": "position_id",

    "close_order_id": "close_order_id",
    "closeorderid": "close_order_id",

    "open_order_id": "open_order_id",
    "openorderid": "open_order_id",

    "user_group_id": "user_group_id",
    "usergroupid": "user_group_id",

    "open_date_time": "open_date_time",
    "opendatetime": "open_date_time",

    "close_date_time": "close_date_time",
    "closedatetime": "close_date_time",

    "lot_size": "lot_size",
    "lotsize": "lot_size",

    "duration_sec": "duration_sec",
    "durationsec": "duration_sec",

    "profit": "profit",

    "reverse_profit": "reverse_profit",
    "reverseprofit": "reverse_profit",

    "net_profit": "net_profit",
    "netprofit": "net_profit",

    "commission": "commission",
    "swap": "swap",
    "amount": "amount",

    "open_price": "open_price",
    "openprice": "open_price",

    "close_price": "close_price",
    "closeprice": "close_price",

    "sl_price": "sl_price",
    "slprice": "sl_price",

    "tp_price": "tp_price",
    "tpprice": "tp_price",

    "open_trade_cross_price":
        "open_trade_cross_price",
    "opentradecrossprice":
        "open_trade_cross_price",

    "close_trade_cross_price":
        "close_trade_cross_price",
    "closetradecrossprice":
        "close_trade_cross_price",

    "side": "side",
    "currency": "currency",

    "campaign_id": "source_campaign_id",
    "campaignid": "source_campaign_id",
}

In [4]:
def normalize_column_name(
    column,
):
    """Convert a raw column name to lowercase snake case."""

    column_name = str(
        column
    ).strip()

    column_name = re.sub(
        r"([A-Z]+)([A-Z][a-z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )

    column_name = re.sub(
        r"_+",
        "_",
        column_name,
    )

    return (
        column_name
        .strip("_")
        .lower()
    )

In [5]:
def parse_campaign_date(
    date_string,
):
    """Parse the campaign date encoded in a filename."""

    normalized_date_string = re.sub(
        r"[_\s]+",
        " ",
        date_string.strip(),
    )

    for date_format in [
        "%d %b %Y",
        "%d %B %Y",
    ]:
        parsed_date = pd.to_datetime(
            normalized_date_string,
            format=date_format,
            errors="coerce",
        )

        if pd.notna(
            parsed_date
        ):
            return parsed_date

    return pd.NaT

In [6]:
def parse_filename(
    path,
):
    """Extract campaign metadata from a raw trade filename."""

    path = Path(
        path
    )

    filename = (
        path.stem
    )

    match = (
        FILENAME_PATTERN
        .search(
            filename
        )
    )

    if match:
        return {
            "campaign_id":
                int(
                    match.group(1)
                ),
            "campaign_date":
                parse_campaign_date(
                    match.group(2)
                ),
        }

    campaign_match = re.search(
        r"Campaign[_\s]*(\d+)",
        filename,
        re.IGNORECASE,
    )

    campaign_id = (
        int(
            campaign_match.group(1)
        )
        if campaign_match
        else None
    )

    return {
        "campaign_id":
            campaign_id,
        "campaign_date":
            pd.NaT,
    }

In [7]:
def load_trade_file(
    path,
):
    """Load and standardize one raw campaign trade file."""

    path = Path(
        path
    )

    metadata = (
        parse_filename(
            path
        )
    )

    if (
        metadata[
            "campaign_id"
        ]
        is None
    ):
        raise ValueError(
            "Could not determine campaign ID "
            f"from {path.name}."
        )

    if (
        path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            path,
            low_memory=False,
        )

    elif (
        path.suffix.lower()
        == ".xlsx"
    ):
        df = pd.read_excel(
            path
        )

    else:
        raise ValueError(
            f"Unsupported file type: "
            f"{path.suffix}"
        )

    normalized_columns = {
        column:
            normalize_column_name(
                column
            )
        for column in df.columns
    }

    df = df.rename(
        columns=normalized_columns
    )

    df = df.rename(
        columns=TRADE_COLUMN_ALIASES
    )

    df.insert(
        0,
        "source_row_number",
        np.arange(
            2,
            len(df) + 2,
        ),
    )

    required_columns = {
        "account_id",
        "open_date_time",
        "close_date_time",
        "amount",
        "net_profit",
        "side",
    }

    missing_required = (
        required_columns
        - set(
            df.columns
        )
    )

    if missing_required:
        raise ValueError(
            f"{path.name} is missing "
            "required columns: "
            f"{sorted(missing_required)}"
        )

    header_echo_mask = (
        df[
            "account_id"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "account_id",
                "accountid",
                "account",
            }
        )
        .fillna(False)
        |
        df[
            "open_date_time"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "open_date_time",
                "opendatetime",
            }
        )
        .fillna(False)
    )

    df = (
        df.loc[
            ~header_echo_mask
        ]
        .copy()
    )

    for column in (
        TRADE_ID_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace(
                    {
                        "":
                            pd.NA,
                        "nan":
                            pd.NA,
                        "None":
                            pd.NA,
                    }
                )
            )

    for column in (
        TRADE_NUMERIC_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    for column in (
        TRADE_DATETIME_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_datetime(
                df[column],
                errors="coerce",
                utc=True,
            )

    df[
        "side"
    ] = (
        df[
            "side"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    if (
        "currency"
        in df.columns
    ):
        df[
            "currency"
        ] = (
            df[
                "currency"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    df[
        "campaign_id"
    ] = metadata[
        "campaign_id"
    ]

    df[
        "campaign_date"
    ] = metadata[
        "campaign_date"
    ]

    df[
        "source_file"
    ] = path.name

    df[
        "source_path"
    ] = str(
        path
    )

    return (
        df.reset_index(
            drop=True
        )
    )

In [8]:
def load_trade_directory(
    directory,
    is_unseen,
):
    """Load every supported trade file in a directory."""

    directory = Path(
        directory
    )

    dataset_label = (
        "UNSEEN"
        if is_unseen
        else "HISTORICAL"
    )

    print(
        f"\nScanning {dataset_label.lower()} "
        f"trade directory:"
    )
    print(
        f"  {directory}"
    )

    if not directory.exists():
        raise FileNotFoundError(
            f"Trade directory not found: "
            f"{directory.resolve()}"
        )

    paths = sorted(
        [
            path
            for path in directory.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in {
                    ".csv",
                    ".xlsx",
                }
            )
        ]
    )

    if not paths:
        raise ValueError(
            "No CSV or XLSX trade files "
            f"found in {directory}."
        )

    print(
        f"Found {len(paths)} "
        f"{dataset_label.lower()} files."
    )

    frames = []

    for file_number, path in enumerate(
        paths,
        start=1,
    ):
        print(
            f"  [{file_number}/{len(paths)}] "
            f"Processing: {path.name}"
        )

        frame = (
            load_trade_file(
                path
            )
        )

        frame[
            "_is_unseen"
        ] = bool(
            is_unseen
        )

        frames.append(
            frame
        )

    combined = pd.concat(
        frames,
        ignore_index=True,
    )

    print(
        f"Loaded {len(combined):,} "
        f"{dataset_label.lower()} trades."
    )

    return combined

In [9]:
def attach_previous_completed_trade(
    trades,
):
    """Attach the most recent completed trade known at entry time."""

    result_frames = []

    grouping_columns = [
        "campaign_id",
        "account_id",
    ]

    for _, group in (
        trades.groupby(
            grouping_columns,
            dropna=False,
            sort=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "close_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        completed = (
            group.loc[
                group[
                    "close_date_time"
                ].notna()
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        previous_columns = [
            "close_date_time",
            "net_profit",
            "amount",
        ]

        if (
            "position_id"
            in completed.columns
        ):
            previous_columns.append(
                "position_id"
            )

        previous = (
            completed[
                previous_columns
            ]
            .rename(
                columns={
                    "close_date_time":
                        (
                            "previous_completed_"
                            "close_date_time"
                        ),
                    "net_profit":
                        (
                            "previous_completed_"
                            "net_profit"
                        ),
                    "amount":
                        (
                            "previous_completed_"
                            "amount"
                        ),
                    "position_id":
                        (
                            "previous_completed_"
                            "position_id"
                        ),
                }
            )
        )

        merged = pd.merge_asof(
            current.sort_values(
                "open_date_time"
            ),
            previous.sort_values(
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "previous_completed_"
                "close_date_time"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        result_frames.append(
            merged
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "previous_completed_was_loss"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        < 0
    )

    result[
        "previous_completed_was_win"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        > 0
    )

    result[
        "reentry_gap_minutes"
    ] = (
        (
            result[
                "open_date_time"
            ]
            - result[
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ]
        )
        .dt.total_seconds()
        / 60
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

In [10]:
def assign_trade_ideas(
    trades,
    maximum_gap_minutes=3.0,
):
    """Assign the Stage 1 directional trade-idea identifier."""

    result = (
        trades.copy()
    )

    group_columns = [
        "account_id",
        "campaign_id",
        "side",
    ]

    result = (
        result
        .sort_values(
            group_columns
            + [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    idea_numbers = pd.Series(
        index=result.index,
        dtype="Int64",
    )

    for _, group in (
        result.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        group = group.sort_values(
            [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )

        current_idea_number = 1
        current_idea_latest_close = (
            pd.NaT
        )

        for row in (
            group.itertuples()
        ):
            if pd.isna(
                current_idea_latest_close
            ):
                current_idea_latest_close = (
                    row.close_date_time
                )

            else:
                gap_minutes = (
                    (
                        row.open_date_time
                        - current_idea_latest_close
                    )
                    .total_seconds()
                    / 60
                )

                if (
                    gap_minutes
                    > maximum_gap_minutes
                ):
                    current_idea_number += 1

                    current_idea_latest_close = (
                        row.close_date_time
                    )

                else:
                    current_idea_latest_close = max(
                        current_idea_latest_close,
                        row.close_date_time,
                    )

            idea_numbers.loc[
                row.Index
            ] = (
                current_idea_number
            )

    result[
        "idea_number"
    ] = (
        idea_numbers
    )

    result[
        "idea_id"
    ] = (
        result[
            "account_id"
        ].astype(str)
        + "_"
        + result[
            "campaign_id"
        ].astype(str)
        + "_"
        + result[
            "side"
        ].astype(str)
        + "_"
        + result[
            "idea_number"
        ].astype(str)
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

In [11]:
def build_stage1_inference_tables(
    historical_trades_dir,
    unseen_trades_dir,
):
    """Build the Stage 1 tables required by the frozen model.

    Historical and unseen trades are processed together so that
    pre-trade historical features for unseen trades can use information
    that would already have been available before the unseen trade opened.

    Args:
        historical_trades_dir: Directory containing known historical trades.
        unseen_trades_dir: Directory containing the hidden/unseen trades.

    Returns:
        Dictionary containing standardized trade tables and idea-level data.
    """

    historical_trades = (
        load_trade_directory(
            directory=(
                historical_trades_dir
            ),
            is_unseen=False,
        )
    )

    unseen_trades = (
        load_trade_directory(
            directory=(
                unseen_trades_dir
            ),
            is_unseen=True,
        )
    )

    trades = pd.concat(
        [
            historical_trades,
            unseen_trades,
        ],
        ignore_index=True,
    )

    trades = (
        trades
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "source_row_number",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    trades[
        "trade_row_id"
    ] = np.arange(
        1,
        len(trades) + 1,
    )

    trades_with_previous_completed = (
        attach_previous_completed_trade(
            trades
        )
    )

    trades_with_ideas = (
        assign_trade_ideas(
            trades=trades,
            maximum_gap_minutes=3.0,
        )
    )

    trades_with_ideas = (
        trades_with_ideas
        .sort_values(
            [
                "idea_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    trades_with_ideas[
        "trade_number_within_idea"
    ] = (
        trades_with_ideas
        .groupby(
            "idea_id"
        )
        .cumcount()
        + 1
    )

    idea_features = (
        trades_with_ideas
        .groupby(
            "idea_id",
            as_index=False,
        )
        .agg(
            account_id=(
                "account_id",
                "first",
            ),
            campaign_id=(
                "campaign_id",
                "first",
            ),
            side=(
                "side",
                "first",
            ),
            idea_start_time=(
                "open_date_time",
                "min",
            ),
            idea_end_time=(
                "close_date_time",
                "max",
            ),
            total_amount=(
                "amount",
                "sum",
            ),
            total_net_profit=(
                "net_profit",
                "sum",
            ),
        )
    )

    idea_features[
        "is_profitable_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        > 0
    )

    idea_features[
        "is_losing_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        < 0
    )

    idea_features[
        "is_breakeven_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        == 0
    )

    unseen_trade_row_ids = (
        trades.loc[
            trades[
                "_is_unseen"
            ],
            "trade_row_id",
        ]
        .tolist()
    )

    return {
        "trades":
            trades,
        "trades_with_previous_completed":
            (
                trades_with_previous_completed
            ),
        "trades_with_ideas":
            trades_with_ideas,
        "idea_features":
            idea_features,
        "unseen_trade_row_ids":
            unseen_trade_row_ids,
    }

## 3. Stage 2 — Leakage-Safe Behavioural Feature Engineering

These functions recreate the behavioural features used by the frozen model, including post-loss behaviour, historical/recent activity, trade timing, challenge state, position sizing, and historical idea performance.

In [12]:
def attach_past_event_count(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    output_column,
):
    """Attach the number of strictly earlier events."""

    event_counts = (
        event_table
        .groupby(
            group_columns
            + [
                event_time_column
            ],
            dropna=False,
        )
        .size()
        .rename(
            "_events_at_timestamp"
        )
        .reset_index()
    )

    event_counts = (
        event_counts
        .sort_values(
            group_columns
            + [
                event_time_column
            ]
        )
    )

    event_counts[
        output_column
    ] = (
        event_counts
        .groupby(
            group_columns,
            dropna=False,
        )[
            "_events_at_timestamp"
        ]
        .cumsum()
    )

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_counts.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_counts[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_counts[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_counts.loc[
                event_mask,
                [
                    event_time_column,
                    output_column,
                ],
            ]
            .sort_values(
                event_time_column
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                output_column
            ] = 0

            output_groups.append(
                current_group
            )

            continue

        merge_time_column = (
            f"_past_{output_column}_time"
        )

        group_events = (
            group_events.rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = (
            pd.merge_asof(
                left=(
                    current_group
                ),
                right=(
                    group_events
                ),
                left_on=(
                    target_time_column
                ),
                right_on=(
                    merge_time_column
                ),
                direction="backward",
                allow_exact_matches=False,
            )
        )

        current_group[
            output_column
        ] = (
            current_group[
                output_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

In [13]:
def attach_rolling_event_counts(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    window_minutes,
    feature_prefix,
):
    """Attach counts of strictly earlier events in rolling windows."""

    result_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask,
                event_time_column,
            ]
            .dropna()
            .sort_values()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        target_times = (
            current_group[
                target_time_column
            ]
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        event_times = (
            group_events
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        for window in (
            window_minutes
        ):
            left_boundaries = (
                target_times
                - np.timedelta64(
                    window,
                    "m",
                )
            )

            left_indices = (
                np.searchsorted(
                    event_times,
                    left_boundaries,
                    side="left",
                )
            )

            right_indices = (
                np.searchsorted(
                    event_times,
                    target_times,
                    side="left",
                )
            )

            feature_column = (
                f"{feature_prefix}"
                f"_past_{window}_minutes"
            )

            current_group[
                feature_column
            ] = (
                right_indices
                - left_indices
            )

        result_groups.append(
            current_group
        )

    return pd.concat(
        result_groups,
        ignore_index=True,
    )

In [14]:
def attach_entry_spacing_features(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    tie_breaker_columns,
    previous_gap_column,
    past_median_gap_column,
    gap_ratio_column,
):
    """Attach pre-entry gap and historical median pace features."""

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask
            ]
            .sort_values(
                [
                    event_time_column
                ]
                + tie_breaker_columns
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                previous_gap_column
            ] = np.nan

            current_group[
                past_median_gap_column
            ] = np.nan

            current_group[
                gap_ratio_column
            ] = np.nan

            output_groups.append(
                current_group
            )

            continue

        event_gap_column = (
            "_entry_gap_minutes"
        )

        group_events[
            event_gap_column
        ] = (
            group_events[
                event_time_column
            ]
            .diff()
            .dt.total_seconds()
            / 60
        )

        group_events[
            past_median_gap_column
        ] = (
            group_events[
                event_gap_column
            ]
            .expanding(
                min_periods=1
            )
            .median()
        )

        merge_time_column = (
            "_previous_event_time"
        )

        events_for_merge = (
            group_events[
                [
                    event_time_column,
                    past_median_gap_column,
                ]
            ]
            .rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                events_for_merge
            ),
            left_on=(
                target_time_column
            ),
            right_on=(
                merge_time_column
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            previous_gap_column
        ] = (
            (
                current_group[
                    target_time_column
                ]
                - current_group[
                    merge_time_column
                ]
            )
            .dt.total_seconds()
            / 60
        )

        valid_ratio = (
            current_group[
                previous_gap_column
            ].ge(0)
            & current_group[
                past_median_gap_column
            ].gt(0)
        )

        current_group[
            gap_ratio_column
        ] = np.where(
            valid_ratio,
            (
                current_group[
                    previous_gap_column
                ]
                / current_group[
                    past_median_gap_column
                ]
            ),
            np.nan,
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

In [15]:
def attach_realized_challenge_state(
    target_table,
    realized_events,
):
    """Attach cumulative realized P&L known before each trade."""

    result_frames = []

    group_columns = [
        "campaign_id",
        "account_id",
    ]

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                realized_events.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    realized_events[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    realized_events[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            realized_events.loc[
                event_mask,
                [
                    "close_date_time",
                    "cumulative_realized_pnl",
                ],
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                "open_date_time"
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                "realized_pnl_before_trade"
            ] = 0.0

            result_frames.append(
                current_group
            )

            continue

        group_events = (
            group_events.rename(
                columns={
                    "close_date_time":
                        (
                            "latest_realized_"
                            "close_before_trade"
                        )
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                group_events
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "latest_realized_"
                "close_before_trade"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        current_group[
            "realized_pnl_before_trade"
        ] = (
            current_group[
                "cumulative_realized_pnl"
            ]
            .fillna(0.0)
        )

        current_group = (
            current_group.drop(
                columns=[
                    "cumulative_realized_pnl"
                ],
                errors="ignore",
            )
        )

        result_frames.append(
            current_group
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )

In [16]:
def attach_historical_position_size_features(
    target_table,
):
    """Attach historical median position-size features."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        medians = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            medians.append(
                np.median(
                    historical_amounts
                )
                if historical_amounts
                else np.nan
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "historical_median_amount_before_trade"
        ] = medians

        result_frames.append(
            current
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "current_to_historical_median_amount_ratio"
    ] = (
        result[
            "amount"
        ]
        / result[
            "historical_median_amount_before_trade"
        ]
    )

    return result

In [17]:
def attach_historical_sizing_consistency_features(
    target_table,
):
    """Attach historical position-size coefficient of variation."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        past_cvs = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            past_count = len(
                historical_amounts
            )

            past_mean = (
                np.mean(
                    historical_amounts
                )
                if past_count >= 1
                else np.nan
            )

            past_std = (
                np.std(
                    historical_amounts,
                    ddof=0,
                )
                if past_count >= 2
                else np.nan
            )

            past_cv = (
                past_std / past_mean
                if (
                    past_count >= 2
                    and past_mean > 0
                )
                else np.nan
            )

            past_cvs.append(
                past_cv
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "past_amount_cv"
        ] = (
            past_cvs
        )

        result_frames.append(
            current
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )

In [18]:
def build_historical_performance_features(
    idea_table,
):
    """Build historical completed-idea performance features."""

    idea_performance = (
        idea_table[
            [
                "idea_id",
                "account_id",
                "campaign_id",
                "idea_start_time",
                "idea_end_time",
                "total_amount",
                "total_net_profit",
                "is_profitable_idea",
                "is_losing_idea",
                "is_breakeven_idea",
            ]
        ]
        .copy()
    )

    idea_performance[
        "idea_profit_per_lot"
    ] = (
        idea_performance[
            "total_net_profit"
        ]
        / idea_performance[
            "total_amount"
        ]
    )

    idea_performance[
        "_win_count"
    ] = (
        idea_performance[
            "is_profitable_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_loss_count"
    ] = (
        idea_performance[
            "is_losing_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_idea_count"
    ] = 1

    idea_performance[
        "_win_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .where(
            idea_performance[
                "is_profitable_idea"
            ],
            0.0,
        )
    )

    idea_performance[
        "_loss_abs_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .abs()
        .where(
            idea_performance[
                "is_losing_idea"
            ],
            0.0,
        )
    )

    events = (
        idea_performance
        .groupby(
            [
                "account_id",
                "idea_end_time",
            ],
            as_index=False,
        )
        .agg(
            completed_idea_count=(
                "_idea_count",
                "sum",
            ),
            completed_win_count=(
                "_win_count",
                "sum",
            ),
            completed_loss_count=(
                "_loss_count",
                "sum",
            ),
            completed_win_profit_sum=(
                "_win_profit",
                "sum",
            ),
            completed_loss_abs_profit_sum=(
                "_loss_abs_profit",
                "sum",
            ),
        )
        .sort_values(
            [
                "account_id",
                "idea_end_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    cumulative_mapping = {
        "past_completed_idea_count":
            "completed_idea_count",
        "past_winning_idea_count":
            "completed_win_count",
        "past_losing_idea_count":
            "completed_loss_count",
        "past_win_profit_sum":
            "completed_win_profit_sum",
        "past_loss_abs_profit_sum":
            (
                "completed_loss_"
                "abs_profit_sum"
            ),
    }

    for (
        output_column,
        source_column,
    ) in (
        cumulative_mapping.items()
    ):
        events[
            output_column
        ] = (
            events
            .groupby(
                "account_id"
            )[
                source_column
            ]
            .cumsum()
        )

    current_ideas = (
        idea_performance[
            [
                "idea_id",
                "account_id",
                "campaign_id",
                "idea_start_time",
            ]
        ]
        .sort_values(
            [
                "idea_start_time",
                "account_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    events = (
        events
        .sort_values(
            [
                "idea_end_time",
                "account_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    historical = pd.merge_asof(
        current_ideas,
        events[
            [
                "account_id",
                "idea_end_time",
                "past_completed_idea_count",
                "past_winning_idea_count",
                "past_losing_idea_count",
                "past_win_profit_sum",
                "past_loss_abs_profit_sum",
            ]
        ],
        left_on=(
            "idea_start_time"
        ),
        right_on=(
            "idea_end_time"
        ),
        by="account_id",
        direction="backward",
        allow_exact_matches=False,
    )

    historical[
        "past_idea_win_rate"
    ] = (
        historical[
            "past_winning_idea_count"
        ]
        / historical[
            "past_completed_idea_count"
        ]
    )

    historical[
        "past_mean_win_profit_per_lot"
    ] = (
        historical[
            "past_win_profit_sum"
        ]
        / historical[
            "past_winning_idea_count"
        ]
    )

    historical[
        "past_mean_loss_abs_profit_per_lot"
    ] = (
        historical[
            "past_loss_abs_profit_sum"
        ]
        / historical[
            "past_losing_idea_count"
        ]
    )

    historical[
        "past_payoff_ratio"
    ] = (
        historical[
            "past_mean_win_profit_per_lot"
        ]
        / historical[
            "past_mean_loss_abs_profit_per_lot"
        ]
    )

    return historical

In [19]:
def build_model_features(
    stage1_tables,
):
    """Recreate the 22 pre-trade features used by the frozen model."""

    trades = (
        stage1_tables[
            "trades"
        ]
        .copy()
    )

    trades_with_ideas = (
        stage1_tables[
            "trades_with_ideas"
        ]
        .copy()
    )

    previous_completed = (
        stage1_tables[
            "trades_with_previous_completed"
        ]
        .copy()
    )

    idea_features = (
        stage1_tables[
            "idea_features"
        ]
        .copy()
    )

    trade_features = (
        trades_with_ideas
        .copy()
    )

    # --------------------------------------------------------
    # Loss response
    # --------------------------------------------------------

    previous_columns = [
        "trade_row_id",
        "previous_completed_close_date_time",
        "previous_completed_net_profit",
        "previous_completed_amount",
        "previous_completed_was_loss",
        "previous_completed_was_win",
        "reentry_gap_minutes",
    ]

    trade_features = (
        trade_features.merge(
            previous_completed[
                previous_columns
            ],
            on="trade_row_id",
            how="left",
            validate="one_to_one",
        )
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "amount"
        ]
        / trade_features[
            "previous_completed_amount"
        ]
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    trade_features[
        "post_loss_entry"
    ] = (
        trade_features[
            "previous_completed_was_loss"
        ]
        .fillna(False)
        .astype(bool)
    )

    trade_features[
        "post_loss_reentry_gap_minutes"
    ] = (
        trade_features[
            "reentry_gap_minutes"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    trade_features[
        "post_loss_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    # --------------------------------------------------------
    # Activity
    # --------------------------------------------------------

    trade_features = (
        attach_past_event_count(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            output_column=(
                "past_trade_open_count"
            ),
        )
    )

    trade_features[
        "first_trade_open_date_time"
    ] = (
        trade_features
        .groupby(
            [
                "account_id",
                "campaign_id",
            ]
        )[
            "open_date_time"
        ]
        .transform(
            "min"
        )
    )

    trade_features[
        "elapsed_active_hours"
    ] = (
        (
            trade_features[
                "open_date_time"
            ]
            - trade_features[
                "first_trade_open_date_time"
            ]
        )
        .dt.total_seconds()
        / 3600
    )

    valid_elapsed = (
        trade_features[
            "elapsed_active_hours"
        ]
        > 0
    )

    trade_features[
        "past_trades_opened_per_active_hour"
    ] = np.where(
        valid_elapsed,
        (
            trade_features[
                "past_trade_open_count"
            ]
            / trade_features[
                "elapsed_active_hours"
            ]
        ),
        np.nan,
    )

    trade_features = (
        attach_rolling_event_counts(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            window_minutes=[
                30,
            ],
            feature_prefix=(
                "trades_opened"
            ),
        )
    )

    # --------------------------------------------------------
    # Trade timing
    # --------------------------------------------------------

    trade_features = (
        attach_entry_spacing_features(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            tie_breaker_columns=[
                "trade_row_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_trade_open"
            ),
            past_median_gap_column=(
                "past_median_trade_open_gap_minutes"
            ),
            gap_ratio_column=(
                "trade_open_gap_to_past_median_ratio"
            ),
        )
    )

    idea_start_events = (
        idea_features[
            [
                "account_id",
                "campaign_id",
                "idea_id",
                "idea_start_time",
            ]
        ]
        .drop_duplicates(
            subset=[
                "idea_id"
            ]
        )
        .copy()
    )

    idea_features = (
        attach_entry_spacing_features(
            target_table=(
                idea_features
            ),
            event_table=(
                idea_start_events
            ),
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "idea_start_time"
            ),
            event_time_column=(
                "idea_start_time"
            ),
            tie_breaker_columns=[
                "idea_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_idea_start"
            ),
            past_median_gap_column=(
                "past_median_idea_start_gap_minutes"
            ),
            gap_ratio_column=(
                "idea_start_gap_to_past_median_ratio"
            ),
        )
    )

    trade_features = (
        trade_features.merge(
            idea_features[
                [
                    "idea_id",
                    "minutes_since_previous_idea_start",
                    "past_median_idea_start_gap_minutes",
                    "idea_start_gap_to_past_median_ratio",
                ]
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # --------------------------------------------------------
    # Challenge state
    # --------------------------------------------------------

    realized_events = (
        trades[
            [
                "account_id",
                "campaign_id",
                "close_date_time",
                "net_profit",
            ]
        ]
        .dropna(
            subset=[
                "close_date_time"
            ]
        )
        .groupby(
            [
                "account_id",
                "campaign_id",
                "close_date_time",
            ],
            as_index=False,
        )[
            "net_profit"
        ]
        .sum()
        .rename(
            columns={
                "net_profit":
                    "realized_pnl_at_close"
            }
        )
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "close_date_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    realized_events[
        "cumulative_realized_pnl"
    ] = (
        realized_events
        .groupby(
            [
                "campaign_id",
                "account_id",
            ]
        )[
            "realized_pnl_at_close"
        ]
        .cumsum()
    )

    trade_features = (
        attach_realized_challenge_state(
            target_table=(
                trade_features
            ),
            realized_events=(
                realized_events
            ),
        )
    )

    trade_features[
        "realized_distance_to_drawdown_limit"
    ] = (
        trade_features[
            "realized_pnl_before_trade"
        ]
        - REALIZED_DRAWDOWN_BOUNDARY
    )

    # --------------------------------------------------------
    # Position sizing
    # --------------------------------------------------------

    trade_features = (
        attach_historical_position_size_features(
            trade_features
        )
    )

    trade_features = (
        attach_historical_sizing_consistency_features(
            trade_features
        )
    )

    # --------------------------------------------------------
    # Historical idea performance
    # --------------------------------------------------------

    historical_idea_features = (
        build_historical_performance_features(
            idea_features
        )
    )

    historical_columns = [
        "idea_id",
        "past_completed_idea_count",
        "past_idea_win_rate",
        "past_mean_win_profit_per_lot",
        "past_mean_loss_abs_profit_per_lot",
        "past_payoff_ratio",
    ]

    trade_features = (
        trade_features.merge(
            historical_idea_features[
                historical_columns
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Replace invalid numerical values exactly as missing values.
    trade_features = (
        trade_features.replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    if (
        "reverse_profit"
        in trade_features.columns
        and "amount"
        in trade_features.columns
    ):
        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit"
            ]
            / trade_features[
                "amount"
            ]
        )

        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit_per_lot"
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
        )

    return (
        trade_features
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

## 4. Frozen Model Inference

The final model is Experiment 10B: a LightGBM regression model using 22 leakage-safe pre-trade behavioural features. The feature order and numerical decision threshold are loaded from `models/exp10b_model_metadata.json`.

Before prediction, the pipeline checks that banned post-trade variables are not part of the frozen model feature set.

In [20]:
def run_frozen_model(
    historical_trades_dir,
    unseen_trades_dir,
    model_path,
    metadata_path,
    output_path=None,
):
    """Run the frozen Stage 3 model and return one decision per unseen trade.

    Args:
        historical_trades_dir: Directory containing historical trades
            available before the unseen period.
        unseen_trades_dir: Directory containing unseen trade files.
        model_path: Path to the frozen model.
        metadata_path: Path to the frozen model metadata.
        output_path: Optional CSV path for exporting decisions.

    Returns:
        Tuple containing:
            - DataFrame with one prediction and fade decision per unseen trade.
            - Full feature table for optional evaluation/debugging.
    """

    print(
        "=" * 70
    )
    print(
        "FROZEN MODEL INFERENCE"
    )
    print(
        "=" * 70
    )

    # --------------------------------------------------------
    # 1. Stage 1 preprocessing
    # --------------------------------------------------------

    print(
        "\n[1/6] Building Stage 1 inference tables..."
    )

    stage1_tables = (
        build_stage1_inference_tables(
            historical_trades_dir=(
                historical_trades_dir
            ),
            unseen_trades_dir=(
                unseen_trades_dir
            ),
        )
    )

    print(
        "Stage 1 preprocessing completed."
    )

    print(
        "  Total trades:",
        f"{len(stage1_tables['trades']):,}",
    )

    print(
        "  Unseen trades:",
        f"{len(stage1_tables['unseen_trade_row_ids']):,}",
    )

    # --------------------------------------------------------
    # 2. Stage 2 feature engineering
    # --------------------------------------------------------

    print(
        "\n[2/6] Building model features..."
    )

    feature_table = (
        build_model_features(
            stage1_tables
        )
    )

    print(
        "Feature engineering completed."
    )

    print(
        "  Feature-table rows:",
        f"{len(feature_table):,}",
    )

    # --------------------------------------------------------
    # 3. Load frozen model specification
    # --------------------------------------------------------

    print(
        "\n[3/6] Loading frozen model..."
    )

    model = joblib.load(
        model_path
    )

    print(
        "  Model loaded from:"
    )
    print(
        f"  {model_path}"
    )

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as file:
        metadata = json.load(
            file
        )

    feature_columns = (
        metadata[
            "feature_columns"
        ]
    )

    decision_threshold = (
        float(
            metadata[
                "decision_threshold"
            ]
        )
    )

    print(
        "  Number of frozen features:",
        len(feature_columns),
    )

    print(
        "  Frozen decision threshold:",
        decision_threshold,
    )

    # --------------------------------------------------------
    # 4. Leakage and schema checks
    # --------------------------------------------------------

    print(
        "\n[4/6] Running leakage and schema checks..."
    )

    banned_columns = {
        "profit",
        "net_profit",
        "reverse_profit",
        "reverse_profit_per_lot",
        "sl_price",
        "tp_price",
    }

    leaking_features = (
        set(
            feature_columns
        )
        & banned_columns
    )

    if leaking_features:
        raise RuntimeError(
            "Leakage detected in frozen "
            "feature set: "
            f"{sorted(leaking_features)}"
        )

    missing_features = (
        set(
            feature_columns
        )
        - set(
            feature_table.columns
        )
    )

    if missing_features:
        raise RuntimeError(
            "Required model features "
            "could not be created: "
            f"{sorted(missing_features)}"
        )

    print(
        "Leakage check passed."
    )

    print(
        "Feature-schema check passed."
    )

    X = (
        feature_table[
            feature_columns
        ]
        .copy()
    )

    # --------------------------------------------------------
    # 5. Frozen model prediction
    # --------------------------------------------------------

    print(
        "\n[5/6] Running frozen model inference..."
    )

    scores = model.predict(
        X
    )

    feature_table[
        "predicted_reverse_profit_per_lot"
    ] = scores

    feature_table[
        "decision_threshold"
    ] = (
        decision_threshold
    )

    feature_table[
        "fade_decision"
    ] = (
        feature_table[
            "predicted_reverse_profit_per_lot"
        ]
        >= decision_threshold
    ).astype(int)

    print(
        "Model inference completed."
    )

    # --------------------------------------------------------
    # 6. Keep only unseen trades
    # --------------------------------------------------------

    print(
        "\n[6/6] Preparing unseen-trade decisions..."
    )

    unseen_trade_row_ids = set(
        stage1_tables[
            "unseen_trade_row_ids"
        ]
    )

    results = (
        feature_table.loc[
            feature_table[
                "trade_row_id"
            ].isin(
                unseen_trade_row_ids
            ),
            [
                "trade_row_id",
                "campaign_id",
                "account_id",
                "source_file",
                "source_row_number",
                "open_date_time",
                "predicted_reverse_profit_per_lot",
                "decision_threshold",
                "fade_decision",
            ],
        ]
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # Output validation
    # --------------------------------------------------------

    expected_unseen_rows = (
        len(
            unseen_trade_row_ids
        )
    )

    if (
        len(results)
        != expected_unseen_rows
    ):
        raise RuntimeError(
            "Output row count does not "
            "match unseen input trade count. "
            f"Expected {expected_unseen_rows}, "
            f"received {len(results)}."
        )

    if (
        results[
            "trade_row_id"
        ]
        .duplicated()
        .any()
    ):
        raise RuntimeError(
            "Duplicate trade decisions detected."
        )

    # --------------------------------------------------------
    # Optional export
    # --------------------------------------------------------

    if (
        output_path
        is not None
    ):
        output_path = Path(
            output_path
        )

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        results.to_csv(
            output_path,
            index=False,
        )

        print(
            "\nPredictions exported to:"
        )
        print(
            f"  {output_path}"
        )

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    n_unseen = (
        len(results)
    )

    n_faded = int(
        results[
            "fade_decision"
        ].sum()
    )

    coverage = (
        n_faded / n_unseen
        if n_unseen > 0
        else np.nan
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "INFERENCE COMPLETE"
    )

    print(
        "=" * 70
    )

    print(
        f"Unseen trades:  "
        f"{n_unseen:,}"
    )

    print(
        f"Fade decisions: "
        f"{n_faded:,}"
    )

    print(
        f"Coverage:       "
        f"{coverage:.2%}"
    )

    print(
        "=" * 70
    )

    return (
        results,
        feature_table,
    )

## 5. Run the End-to-End Pipeline

Place historically available files under `data/User Trades/` and the new files to score under `data/Unseen User Trades/`, then run this cell.

Predictions are written to:

```text
results/inference/unseen_trade_decisions.csv
```

In [21]:
(
    unseen_trade_decisions,
    unseen_feature_table,
) = run_frozen_model(
    historical_trades_dir=(
        HISTORICAL_TRADES_DIR
    ),
    unseen_trades_dir=(
        UNSEEN_TRADES_DIR
    ),
    model_path=(
        FROZEN_MODEL_PATH
    ),
    metadata_path=(
        FROZEN_MODEL_METADATA_PATH
    ),
    output_path=(
        PREDICTION_OUTPUT_PATH
    ),
)

display(
    unseen_trade_decisions.head()
)


FROZEN MODEL INFERENCE

[1/6] Building Stage 1 inference tables...

Scanning historical trade directory:
  /Users/charltonsiaw/Desktop/C22-veNTUre/data/User Trades
Found 34 historical files.
  [1/34] Processing: Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv
  [2/34] Processing: Campaign 34 Data 03 Mar 2026 XAUUSD only (1D).csv
  [3/34] Processing: Campaign 35 Data 06 Mar 2026 XAUUSD only (1D).csv
  [4/34] Processing: Campaign 36 Data 09 Mar 2026 XAUUSD only (1D).csv
  [5/34] Processing: Campaign 37 Data 13 Mar 2026 XAUUSD only (1D).csv
  [6/34] Processing: Campaign 38 Data 17 Mar 2026 XAUUSD only (1D).csv
  [7/34] Processing: Campaign 39 Data 20 Mar 2026 XAUUSD only (1D).csv
  [8/34] Processing: Campaign 40 Data 24 Mar 2026 XAUUSD only (1D).csv
  [9/34] Processing: Campaign 41 Data 27 Mar 2026 XAUUSD only (1D).csv
  [10/34] Processing: Campaign 42 Data 31 Mar 2026 XAUUSD only (1D).csv
  [11/34] Processing: Campaign 43 Data 6 Apr 2026 XAUUSD only (1D).csv
  [12/34] Processing: Campa

,trade_row_id,campaign_id,account_id,source_file,source_row_number,open_date_time,predicted_reverse_profit_per_lot,decision_threshold,fade_decision
0,25167,53,D#1589404,Campaign 53 Data 19 May 2026 XAUUSD only (1D).csv,663,2026-05-19 11:46:27+00:00,-51.094511,85.102037,0
1,25169,53,D#1589405,Campaign 53 Data 19 May 2026 XAUUSD only (1D).csv,459,2026-05-19 06:24:14+00:00,7.901907,85.102037,0
2,25171,53,D#1589405,Campaign 53 Data 19 May 2026 XAUUSD only (1D).csv,572,2026-05-19 09:06:22+00:00,108.953362,85.102037,1
3,25173,53,D#1589405,Campaign 53 Data 19 May 2026 XAUUSD only (1D).csv,725,2026-05-19 11:38:18+00:00,-29.656335,85.102037,0
4,25175,53,D#1589405,Campaign 53 Data 19 May 2026 XAUUSD only (1D).csv,808,2026-05-19 13:29:22+00:00,12.458611,85.102037,0


## 6. Inference Summary

In [24]:
pipeline_summary = pd.DataFrame(
    {
        "metric": [
            "n_unseen_trades",
            "n_fade_decisions",
            "coverage",
        ],
        "value": [
            len(
                unseen_trade_decisions
            ),
            int(
                unseen_trade_decisions[
                    "fade_decision"
                ].sum()
            ),
            (
                unseen_trade_decisions[
                    "fade_decision"
                ].mean()
            ),
        ],
    }
)

display(
    pipeline_summary
)


,metric,value
0,n_unseen_trades,21355.000000
1,n_fade_decisions,1980.000000
2,coverage,0.092718


## Notes

- Historical trades are used only to reconstruct information that would have been available before each new trade.
- The frozen model, feature list, and numerical threshold are not changed during inference.
- Realised post-trade outcomes are not required to generate a decision.
- The research notebooks contain the exploratory analysis, feature research, experiment history, backtests, and validation diagnostics.